# Stage-Aware GPU Co-location on Kaggle (2x T4)

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, and Internet **ON**
(needed only to clone and pip install; the workloads themselves all have offline fallbacks).

A Kaggle session is capped at 12 hours and the weekly GPU quota is 30 hours. The full
pairwise sweep needs roughly **7 GPU-hours**, so it fits in one session, but every stage
writes to `data/` and can be resumed separately if you are interrupted.

Run the cells in order. Cell 2 is the important one: it tells you whether this session
can actually measure occupancy, which determines whether the headline ablation is
available to you.

In [ ]:
# 1. Get the code and dependencies.
#    Three sources, tried in order:
#      (a) a Kaggle Dataset you uploaded (no GitHub needed)
#      (b) a git clone, if you set SAGC_REPO_URL below
#      (c) the code already sitting in /kaggle/working
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working/stage-aware-gpu-colocation"
REPO_URL = os.environ.get("SAGC_REPO_URL", "")   # e.g. https://github.com/gadimahi/stage-aware-gpu-colocation.git

def from_dataset():
    hits = glob.glob("/kaggle/input/*/sagc") + glob.glob("/kaggle/input/*/*/sagc")
    if not hits:
        return False
    src = os.path.dirname(hits[0])
    if os.path.isdir(WORK):
        shutil.rmtree(WORK)
    shutil.copytree(src, WORK)
    print("copied from Kaggle Dataset:", src)
    return True

def from_git():
    if not REPO_URL:
        return False
    if not os.path.isdir(WORK):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
    print("cloned from", REPO_URL)
    return True

if not (from_dataset() or from_git() or os.path.isdir(WORK)):
    raise SystemExit(
        "No source found. Either upload the repo as a Kaggle Dataset "
        "(Add Data -> New Dataset -> upload the zip), or set SAGC_REPO_URL.")

os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "lightgbm", "pulp", "pyarrow", "nvidia-ml-py"], check=False)
print("working directory:", os.getcwd())

In [ ]:
# 2. GO / NO-GO. Read this output carefully before spending GPU-hours.
#
#    backend=cuda                 good, results will be MEASURED
#    profiler tier=dcgm           best case: true achieved occupancy available
#    profiler tier=pynvml         degraded: occupancy unavailable, ablation A1 impossible
#    co-location mode=mps         partition control available
#    co-location mode=concurrent  no MPS; thread-percentage sweep collapses to 100%
!python -m sagc probe

In [ ]:
# 3. Sanity-check the code before it touches the GPU in anger (about 10 seconds)
!python -m pytest tests/ -q

In [ ]:
# 4. M2: profile all twelve workloads solo, three repetitions each.
#    Completion criterion: occupancy coefficient of variation below 0.05.
#    Roughly 25 minutes on 2x T4.
!python -m sagc profile --reps 3

In [ ]:
# 5. M3: the pairwise co-location sweep. The expensive stage, roughly 5 GPU-hours.
#    Solo baselines are re-measured within this session, which is not optional:
#    a slowdown ratio against a stale baseline is noise.
#
#    Behind schedule? Drop to a single partition setting and keep the full pair set:
#        !python -m sagc sweep --reps 3 --thread-pcts 100
#    Pair breadth matters more than partition depth, because the generalisation
#    claim lives on breadth.
!python -m sagc sweep --reps 3 --thread-pcts 100,50

In [ ]:
# 6. M4: train the predictor and run ablations A1 to A6.
#    Watch two numbers:
#      leave-one-workload-out MAE   should be well under 0.15
#      A1 counters vs utilisation   counters should win; if they do not, report that
!python -m sagc train

In [ ]:
# 7. M5/M6: policy sweep, self-consistency checks, greedy optimality gap.
#    --bandit adds the contextual-bandit stretch goal (trained in simulation,
#    so it costs no GPU-hours).
!python -m sagc schedule --pipelines 20

In [ ]:
# 8. All eight paper figures
!python -m sagc figures

import glob
from IPython.display import Image, display
for p in sorted(glob.glob("figures/*.png")):
    print(p)
    display(Image(p))

In [ ]:
# 9. A real hardware campaign under exclusive allocation.
#    This is Objective 1: the measured held-but-idle fraction that motivates the
#    whole project. Start small (4 pipelines) and scale up once it works.
from sagc.m1_executor import executor

trace = executor.characterise(n_pipelines=4, verbose=True)
trace.save("results", tag="hardware_exclusive")

In [ ]:
# 10. Validate the simulator against that hardware run.
#     Acceptance criterion: simulated makespan within 10 per cent of measured.
import pandas as pd
from sagc.m6_sim import validate

print(validate.validate(trace, pd.read_parquet("data/pairs.parquet")))

In [ ]:
# 11. Package the artefacts so they survive the session ending
import shutil
for d in ("data", "results", "figures"):
    shutil.make_archive(f"/kaggle/working/{d}", "zip", d)
print("download data.zip / results.zip / figures.zip from the Output pane")